# SO_MAP Tutorial

Author : Henry Nachman

---

The pspy package uses the `so_map` class to manipulate CMB maps. This notebook will explore how to use this class.

In [1]:
import matplotlib
matplotlib.use("nbAgg")
from pspy import so_map, so_window, so_mcm, sph_tools, so_spectra, pspy_utils
import numpy as np
import healpy as hp
import pylab as plt
import os

test_dir = "notebook_results"

In [2]:
lon, lat = 16, -1.3
radius = 15
# nside = 1024
# ncomp=3 mean that we are going to use spin0 and 2 field
ncomp = 3
# specify the order of the spectra, this will be the order used in pspy
# note that if you are doing cross correlation between galaxy and kappa for example, you should follow a similar structure
spectra = ["TT", "TE", "TB", "ET", "BT", "EE", "EB", "BE", "BB"]
# clfile are the camb lensed power spectra
clfile = "/shared_home/henachman/home/software/pspy/tutorials/data/bode_almost_wmap5_lmax_1e4_lensedCls_startAt2.dat"
# n_splits stand for the number of splits we want to simulate
n_splits = 2
# the maximum multipole to consider
lmax = 3000
# the number of iteration in map2alm
niter = 0
# the noise on the spin0 component, if not specified, the noise in polarisation wil be sqrt(2)x that
rms_uKarcmin_T = 20e-6
# the apodisation lengh for the survey mask (in degree)
apo_radius_degree_survey = 1
# the number of holes in the point source mask
source_mask_nholes = 100
# the radius of the holes (in arcminutes)
source_mask_radius = 10
# the apodisation lengh for the point source mask (in degree)
apo_radius_degree_mask = 0.3
# if you want to write down the data file
write_data = False

In [3]:
# create a binningfile with format, lmin,lmax,lmean
pspy_utils.create_binning_file(bin_size=40, n_bins=300, file_name="%s/binning.dat" % test_dir)
binning_file = "%s/binning.dat" % test_dir

Input Map

In [4]:
path_to_fits = "/shared_home/henachman/data/sim_sky_maps/cmb_run_HEALPIX.fits"

# Since the coordinates of the map are given in RA, Dec, we need to specify coordinate as "G"
input_map = so_map.read_map(
    file=path_to_fits,
    coordinate="C",
)
input_map.plot(file_name=f"{test_dir}/input_map") # hp_gnomv=(16.06, -1.28, 5000, 1), 

Load the map

In [4]:
# print(help(so_map.read_map))

path_to_fits = "/shared_home/henachman/data/sim_outputs/CMB/Deep56Coadd/Deep56_uK_fits/filterbin_filtered_map.fits"

# Since the coordinates of the map are given in RA, Dec, we need to specify coordinate as "G"
map = so_map.read_map(
    file=path_to_fits,
    coordinate="C",
)

In [5]:
print(map.pixel)
map.plot(file_name=f"{test_dir}/map") # hp_gnomv=(16.06, -1.28, 5000, 1), 

HEALPIX


Get the map NSIDE

In [6]:
print(map.nside)

8192


To make splits of the map, the input map must be of `so_map` type.

In [7]:
n_splits=1
rms_uKarcmin_T = 20


splitlist = []
nameList = []

# %% MARK: Gen. Splits
print("generate sims")
#let's make 2 splits out of it, each with 'rms_uKarcmin_T' rms in T and 'rms_uKarcmin_T'xsqrt(2) rms in pol
for i in range(n_splits):
    split = map.copy()
    noise = so_map.white_noise(split, rms_uKarcmin_T=rms_uKarcmin_T)
    split.data += noise.data
    splitlist += [split]
    nameList += ["split_%d" % i]
    split.plot(file_name="%s/split_%d" %(test_dir, i))

generate sims
